# 12 Path diagram, finer boxes

For the two path-diagram prompts (36 + 59 and 36 + 53), what the Anthropic-style diagram needs beyond the saved graphs:
1. labels for the features on the operand tokens (what property of the number each tracks);
2. how much of each operand feature's input direction lies along the fast and the slow circles of Q2;
3. the answer with each finer box at "=" switched off.

Saved to `results/path_diagram/`. About 15 minutes. Run as **Save Version -> Save & Run All**, then download `path_diagram_fine.zip`.

Setup: GPU T4, Internet on, secret `HF_TOKEN` ticked.

In [ ]:
!rm -rf helix_addition && git clone https://github.com/amark-23/helix_addition.git
!pip install -q git+https://github.com/safety-research/circuit-tracer@v0.5.3

In [ ]:
import os, sys
os.chdir("/kaggle/working/helix_addition")
sys.path.insert(0, "/kaggle/working/helix_addition/src")
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
import numpy as np, pandas as pd
from helixadd import model as hmodel, diagram, intervene, prompts
model = hmodel.load_model()
data = dict(np.load("results/number_acts/data.npz"))
PAIRS = [(36, 59), (36, 53)]
nodes = {p: pd.read_csv(f"results/path_diagram/{p[0]}_{p[1]}/nodes.csv") for p in PAIRS}

## 1-2. Operand features: labels and circle shares

In [ ]:
rows = []
for position, pos in [("a", diagram.A_POS), ("b", diagram.B_POS)]:
    feats = sorted({(int(r.layer), int(r.feature)) for n in nodes.values()
                    for r in n[(n.kind == "feature") & (n.pos == pos)].itertuples()})
    acts = diagram.operand_feature_acts(model, feats, position)
    lab = diagram.score_operand_features(acts)
    lab.insert(0, "feature", [f for _, f in feats]); lab.insert(0, "layer", [l for l, _ in feats]); lab.insert(0, "position", position)
    shares = diagram.encoder_circle_shares(model, feats, data, position)
    rows.append(lab.merge(shares, on=["layer", "feature"]))
    print(position, len(feats), "features;", lab["label"].value_counts().to_dict())
op = pd.concat(rows)
op.to_csv("results/path_diagram/operand_features.csv", index=False)
op.groupby(["position", "label"])[["fast share", "slow share"]].mean().round(4)

## 3. Each finer box at the = token switched off

In [ ]:
labels = pd.read_csv("results/feature_labels/data.csv")
out = []
for a, b in PAIRS:
    n = nodes[(a, b)]
    eq = n[(n.kind == "feature") & (n.pos == 43)].merge(labels, on=["layer", "feature"])
    eq["box"] = eq["best_var"].map(diagram.EQ_BOX_OF_VARIABLE)
    eq = eq[~eq["role"].isin(["mostly active", "unclear"]) & eq["box"].notna()]
    for box in diagram.EQ_BOXES:
        g = eq[eq["box"] == box]
        if len(g) == 0:
            continue
        text, _ = intervene.suppress_generate(model, [prompts.make_prompt(a, b)], intervene.by_layer(g["layer"], g["feature"]))
        out.append({"a": a, "b": b, "box": box, "label": diagram.eq_box_label(box, a, b), "n features": len(g),
                    "answer": text[0].split(chr(10))[0]})
bi = pd.DataFrame(out)
bi.to_csv("results/path_diagram/box_interventions.csv", index=False)
bi

In [ ]:
!cd /kaggle/working && zip -qr path_diagram_fine.zip helix_addition/results/path_diagram/operand_features.csv helix_addition/results/path_diagram/box_interventions.csv && ls -lh path_diagram_fine.zip